In [43]:
"""
Solenoid summary: peak conductor field, central field, and hoop stress
by thin-shell magnetic pressure and by Wilson thick wall.
"""

import math

MU0 = 4.0e-7 * math.pi
P0 = {0: 1.0, 2: -1/2, 4: 3/8, 6: -5/16, 8: 35/128, 10: -63/256}   # P_n(0)


def _br(g, a, b):                      # [ ... ]_{r=1}^{r=alpha}
    return g(a, b) - g(1.0, b)

def F(a, b):
    return b * math.log((a + math.hypot(a, b)) / (1.0 + math.hypot(1.0, b)))

def FE2(a, b):
    g = lambda r, b: r**3 / (r*r + b*b)**1.5
    return -_br(g, a, b) / (2 * b)

def FE4(a, b):
    g = lambda r, b: r**3*(2*r**4 + 7*r**2*b**2 + 20*b**4) / (r*r + b*b)**3.5
    return -_br(g, a, b) / (24 * b**3)

def FE6(a, b):
    g = lambda r, b: r**3*(8*r**8 + 44*r**6*b**2 + 99*r**4*b**4
                           + 28*r**2*b**6 + 280*b**8) / (r*r + b*b)**5.5
    return -_br(g, a, b) / (240 * b**5)

def FE8(a, b):
    g = lambda r, b: r**3*(16*r**12 + 120*r**10*b**2 + 390*r**8*b**4 + 715*r**6*b**6
                           + 1080*r**4*b**8 - 1008*r**2*b**10
                           + 1344*b**12) / (r*r + b*b)**7.5
    return -_br(g, a, b) / (896 * b**7)

def FE10(a, b):
    g = lambda r, b: r**3*(128*r**16 + 1216*r**14*b**2 + 5168*r**12*b**4
                           + 12920*r**10*b**6 + 20995*r**8*b**8 + 19976*r**6*b**10
                           + 49632*r**4*b**12 - 46464*r**2*b**14
                           + 21120*b**16) / (r*r + b*b)**9.5
    return -_br(g, a, b) / (11520 * b**9)

TERMS = {0: F, 2: FE2, 4: FE4, 6: FE6, 8: FE8, 10: FE10}


# ---------------------------------------------------------------- fields
def b_peak(length, r_inner, r_outer, j, nmax=10):
    """Peak conductor field at (r,z) = (a1,0), Legendre expansion at xi = 1. [T]"""
    a1 = r_inner
    alpha, beta = r_outer / a1, 0.5 * length / a1
    return MU0 * j * a1 * sum(P0[n] * TERMS[n](alpha, beta)
                              for n in sorted(TERMS) if n <= nmax)


def b_center(length, r_inner, r_outer, j):
    """On-axis central field, exact Biot-Savart over the winding cross-section. [T]"""
    zp, zm = 0.5 * length, -0.5 * length

    def log_term(z):
        return z * math.log((math.hypot(r_outer, z) + r_outer) /
                            (math.hypot(r_inner, z) + r_inner))

    return 0.5 * MU0 * j * (log_term(zp) - log_term(zm))


# ---------------------------------------------------------------- stresses
def wilson_peak(length, r_inner, r_outer, j, B1, nu=0.3, kappa=0.0, npts=201):
    """Peak Wilson hoop stress [Pa] at the bore, for a given B1."""
    a1, alpha = r_inner, r_outer / r_inner
    S = j * B1 * a1 / (alpha - 1.0)

    kA = (2 + nu) / 3 * (alpha - kappa)
    kB = (3 + nu) / 8 * (1 - kappa)

    C0 = kA * (alpha**2 + alpha + 1) / (alpha + 1) - kB * (alpha**2 + 1)
    C2 = alpha**2 * (kA / (alpha + 1) - kB)
    C1 = -(1 + 2*nu) / 3 * (alpha - kappa)
    C3 = (1 + 3*nu) / 8 * (1 - kappa)

    return max(S * (C0 + C2 / p**2 + C1 * p + C3 * p**2)
               for p in (1.0 + (alpha - 1.0) * i / (npts - 1) for i in range(npts)))


def summary(length, r_inner, r_outer, j, nu=0.3, kappa=0.0):
    """
    B1     peak conductor field at (a1,0), Legendre expansion   [T]
    B0     central field on axis, Biot-Savart                   [T]
    sig_p  thin-shell magnetic pressure (B0^2/2mu0)(a1/th)      [Pa]
    sig_w  Wilson peak hoop at the bore, using B1               [Pa]
    """
    th = r_outer - r_inner
    B1 = b_peak(length, r_inner, r_outer, j)
    B0 = b_center(length, r_inner, r_outer, j)
    sig_p = (B0**2 / (2.0 * MU0)) * (r_inner / th)
    sig_w = wilson_peak(length, r_inner, r_outer, j, B1, nu, kappa)
    je_ref    = 1e8          # [A/m²]  arbitrary reference (100 A/mm²)
    sigma_ref =750e6       # [Pa]    arbitrary reference (750 MPa)
    # σ ∝ Je²  →  Je_lim = Je_ref * sqrt(sigma_limit / sigma_ref)
    je_lim = je_ref * np.sqrt(sig_p / sigma_ref)


    print(f"\nL = {length*1e3:.1f} mm   a1 = {r_inner*1e3:.1f} mm   "
          f"a2 = {r_outer*1e3:.1f} mm   j = {j/1e6:.1f} A/mm^2")
    print(f"B1  (peak, Legendre)         = {B1:8.4f} T")
    print(f"B0  (centre, Biot-Savart)    = {B0:8.4f} T")
    print(f"sigma_hoop (mag. pressure)   = {sig_p/1e6:8.2f} MPa")
    print(f"sigma_hoop (Wilson, peak)    = {sig_w/1e6:8.2f} MPa")
    print(f"Je_lim (σ_ref = 750 MPa)     = {je_lim/1e6:8.2f} A/mm^2")
    return B1, B0, sig_p, sig_w


summary(length=5, r_inner=0.755, r_outer=0.980875, j=88.6e6)


L = 5000.0 mm   a1 = 755.0 mm   a2 = 980.9 mm   j = 88.6 A/mm^2
B1  (peak, Legendre)         =  23.8964 T
B0  (centre, Biot-Savart)    =  23.7526 T
sigma_hoop (mag. pressure)   =   750.34 MPa
sigma_hoop (Wilson, peak)    =  1058.26 MPa
Je_lim (σ_ref = 750 MPa)     =   100.02 A/mm^2


(23.89644527254631, 23.752570964585708, 750343586.7740873, 1058264038.7779905)